# AdaptAuth-ML: demostración de ejecución

**Modelo de autenticación adaptativa basado en Machine Learning para la detección de accesos anómalos en PYMES del sector de servicios empresariales en Lima, 2026.**

Tesista: Ashly Herminia Jasmín Chávez Marón · Asesor: Ing. Daniel Alfonso Quinto Pazce · UNMSM, Tesis II

Código fuente: [github.com/AshlyChavez/adaptauth-ml](https://github.com/AshlyChavez/adaptauth-ml)

| Paso | Qué se ejecuta | Tiempo aproximado |
|---|---|---|
| 1 | Descarga del código desde GitHub | segundos |
| 2 | Descarga del dataset SuspiciousLogin | segundos |
| 3 | Líneas base: reglas estáticas e Isolation Forest | 1 a 2 min |
| 4 | AdaptAuth-ML apilado, ablación sin IF, pruebas pareadas y bootstrap por usuario | 3 a 6 min |
| 5 | Verificación de requisitos no funcionales (latencias en una sola corrida) | 2 a 4 min |
| 6 | CERT r4.2: historial del empleado, validación cruzada por empleado | varios minutos |

Para ejecutar todo: **Entorno de ejecución → Ejecutar todo**.

## Paso 1. Descarga del código desde GitHub

In [ ]:
import os
os.chdir("/content")
!rm -rf adaptauth-ml
!git clone -q https://github.com/AshlyChavez/adaptauth-ml.git
os.chdir("/content/adaptauth-ml")
!pip -q install shap
print("Estructura del repositorio:")
!ls

## Paso 2. Dataset SuspiciousLogin (Google Workspace)

Inicios de sesión exitosos de dos instituciones durante ocho meses. La etiqueta es la marca de sospecha de Google Workspace.

In [ ]:
import glob
import pandas as pd
from IPython.display import Image, display, Markdown
pd.set_option("display.width", 200)

if not os.path.exists("/content/SL"):
    !git clone -q --depth 1 https://github.com/salomaopena/SuspiciousLoginDataset.git /content/SL
SL = glob.glob("/content/SL/**/suspicious_logins_public_v1.csv", recursive=True)[0]
d = pd.read_csv(SL)
print(f"Eventos: {len(d):,} | Usuarios: {d['actor_pseudo_id'].nunique():,} | Sospechosos: {d['label'].mean()*100:.2f} %")
display(d[["event_hour", "ip_country", "new_ip", "is_new_country", "impossible_travel", "distinct_ips_30d", "logins_30d", "label"]].head())

## Paso 3. Líneas base

- **Reglas estáticas**: IP nueva, país nuevo, viaje imposible, hora atípica.
- **Isolation Forest** aislado.

Partición temporal: entrenamiento de diciembre a marzo, calibración en abril y prueba en mayo y junio.

In [ ]:
!python run_suspiciouslogin.py --data "{SL}"

### Resultados de las líneas base en prueba

In [ ]:
t = pd.read_csv("results_sl/tabla_comparativa_sl.csv")
t = t[t["modelo"].isin(["reglas_estaticas", "iforest"])]
display(t[["modelo", "auc_roc", "auc_pr", "f1_theta1", "precision_theta1", "recall_theta1", "fpr_theta1"]].round(3))

## Paso 4. AdaptAuth-ML apilado y ablación

AdaptAuth-ML es el mismo modelo en ambos datasets: un clasificador HistGradientBoosting que recibe las variables contextuales más el score reescalado de Isolation Forest. Umbrales: θ1 (OTP) maximiza F1 con FPR ≤ 5 %; θ2 (bloqueo) mantiene FPR ≤ 1 %.

Se incluye la **ablación**: el mismo clasificador sin el score de Isolation Forest.

In [ ]:
!python run_analisis_complementario.py --data "{SL}"

In [ ]:
import json
display(Markdown("### Comparación en prueba (meta: F1 ≥ 0.60)"))
display(pd.read_csv("results_complementarios/ablacion_sl.csv").round(3))
display(Image("results_complementarios/curva_pr_modelos_sl.png", width=850))
j = json.load(open("results_complementarios/analisis_complementario.json"))
filas = []
for k, v in j["pareadas_14_grupos"].items():
    filas.append({"comparación": k, "prueba": v["prueba"], "estadístico": round(v["estadistico"], 2),
                  "p": f'{v["p_valor"]:.3g}', "Shapiro p": round(v["shapiro_p_diferencias"], 3)})
display(Markdown("### Pruebas pareadas sobre 14 grupos de usuarios"))
display(pd.DataFrame(filas))
display(Markdown("### Bootstrap por usuario (1 000 réplicas, diferencia de F1)"))
display(pd.DataFrame([{"comparación": k, "dif. media": round(v["dif_media"], 3),
                       "IC 95 %": [round(x, 3) for x in v["ic95"]]} for k, v in j["bootstrap_usuarios"].items()]))
display(Markdown("### Desempeño según historial del usuario"))
display(pd.DataFrame(j["por_historial_apilado"]).T.round(3))

## Paso 5. Verificación de requisitos no funcionales

RNF-04 (inferencia) y RNF-05 (respuesta C1 a C4) se miden en la misma corrida de 1 000 eventos. También se verifica el score de accesos legítimos, el reentrenamiento seguro, la contingencia OTP, el registro de eventos, la explicación de cada alerta (SHAP) y la ausencia de identificadores directos (RNF-12).

In [ ]:
!python run_requisitos_sl.py --data "{SL}"

In [ ]:
display(Markdown("### Matriz de cumplimiento"))
display(pd.read_csv("results_requisitos_sl/matriz_cumplimiento_requisitos.csv"))
display(Markdown("### Distribución de decisiones en prueba"))
reg = pd.read_csv("results_requisitos_sl/registro_eventos_prueba.csv")
display(reg["decision"].value_counts().to_frame("eventos"))
display(Markdown("### Ejemplos de alertas explicadas (top-3 variables)"))
display(pd.read_csv("results_requisitos_sl/explicacion_alertas_top3.csv").sort_values("score_R", ascending=False).head(8))

## Paso 6. CERT Insider Threat r4.2: historial del empleado

Organización de 1 000 empleados (Carnegie Mellon University). Validación cruzada de 5 grupos de empleados: ningún empleado aparece en entrenamiento y prueba a la vez. Las pruebas pareadas usan las predicciones fuera de grupo, con los empleados repartidos al azar en 10 grupos.

Los datos se leen desde **Mi unidad → AdaptAuth-ML → CERT_r4.2**. Si `EJECUTAR_CERT = False`, se muestran los resultados ya guardados en el repositorio.

In [ ]:
EJECUTAR_CERT = True

CERT = "/content/drive/MyDrive/AdaptAuth-ML/CERT_r4.2"
if EJECUTAR_CERT:
    from google.colab import drive
    drive.mount("/content/drive")
    if glob.glob(f"{CERT}/**/logon.csv", recursive=True):
        !python run_cert_v2.py --dir "{CERT}"
    else:
        print("No se encontró CERT_r4.2 en Drive; se muestran los resultados guardados en el repositorio.")

In [ ]:
display(Markdown("### Resultados CERT r4.2 (meta: AUC ≥ 0.90; recall ≥ 0.70 con FPR ≤ 0.1 %)"))
c = pd.read_csv("results_cert_v2/tabla_comparativa_cert_v2.csv")
c["modelo"] = c["modelo"].replace({"hibrido_apilado": "AdaptAuth-ML apilado (con IF)", "supervisado": "Supervisado sin IF",
                                   "hibrido": "Híbrido lineal", "reglas_estaticas": "Reglas estáticas", "iforest": "Isolation Forest"})
display(c[["modelo", "auc_roc", "auc_pr", "f1_theta1", "recall_theta1", "fpr_theta1"]].round(4))
display(Markdown("### Recall según el presupuesto de falsas alarmas (AdaptAuth-ML)"))
display(pd.read_csv("results_cert_v2/puntos_operacion_cert_v2.csv")[["fpr_prueba", "recall", "precision", "alertas_falsas", "dias_detectados"]].round(4))
display(Markdown("### Resultados por unidad funcional"))
display(pd.read_csv("results_cert_v2/por_unidad_cert_v2.csv").round(3))

## Resumen

| Dataset | Resultado principal | Meta |
|---|---|---|
| SuspiciousLogin | F1 0.620 · precisión 0.814 · recall 0.501 · FPR 0.93 % (bloqueo θ2: F1 0.629, FPR 0.46 %) | F1 ≥ 0.60 ✔ |
| CERT r4.2 | AUC-ROC 0.947 · recall 0.70 con FPR 0.11 % | AUC ≥ 0.90 ✔ · recall parcial |

AdaptAuth-ML supera de forma significativa a las reglas estáticas en ambos datasets: t(13) = 37.67 en SuspiciousLogin y t(9) = 7.27 en CERT, p < 0.001. Frente al mismo clasificador sin Isolation Forest no hay diferencia significativa en F1 (p = 0.056 y p = 0.42): el valor está en el componente supervisado, e Isolation Forest aporta una señal desde el primer día.

Código y resultados: [github.com/AshlyChavez/adaptauth-ml](https://github.com/AshlyChavez/adaptauth-ml)